# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 09 · One fixed-tree feature-transfer fold

**Exactly four arms, eight coordinate estimators, one fold now.**

A: existing motion/context control. B: A + full arrival. C: B + repeated terminal pair vectors. D: B + actual observed pair histories. C and D share support masks, terminal ages and retained dimensions. Model settings and exposure are fixed.

These scores are not comparable directly with Round 2 or the historical Kaggle target: sample size and estimator changed. Within this experiment, all arms use identical rows, labels and estimator settings. No old ridge fit, neural job, ensemble or Kaggle submission is run.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round3')
OUT = Path('/home/sagemaker-user/nfl-feature-round3-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Upload/extract the kit in the existing NFL space first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage, online=False):
    cmd=[str(PY),str(KIT/'run_round.py'),stage]
    if online: cmd.append('--online')
    process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout: print(line,end='')
        code=process.wait()
    except KeyboardInterrupt:
        process.send_signal(2); process.wait(timeout=10); raise
    if code:
        raise RuntimeError(f'{stage} stopped with exit {code}. Preserve outputs; inspect the log and export the report. Do not change hyperparameters.')
def show(fig,name):
    visuals.save(fig,OUT,name).show()


## First fold only
The runner checkpoints every 30 boosting iterations, with a 480-second command cap. It never uses validation to select the iteration. Completed models are reused on an unchanged restart. No command here starts follow-up folds.

In [ ]:
assert visuals.load(OUT/'preparation.json')['status']=='expanded_features_ready'
assert visuals.load(OUT/'runtime.json')['status']=='tree_runtime_ready'
run('fit')

In [ ]:
show(visuals.first_fold(OUT),'first_fold_rmse')
show(visuals.contrasts(OUT),'paired_contrasts')

In [ ]:
show(visuals.slices(OUT,'horizon'),'horizon_errors')
show(visuals.slices(OUT,'role'),'role_errors')

## Fresh-process replay, then report
This stage is prohibited from fitting missing work. A failed replay is a debugging stop, not permission to retrain the control.

In [ ]:
run('replay')
result=visuals.load(OUT/'fold_1/summary.json')
print(json.dumps({'continue_eligible':result['continue_research'],'contrasts':result['contrasts']},indent=2))
run('report')
print('Download:',OUT/'nfl_feature_round3_report.zip')

## Decision boundary
If all contrasts fail, stop this interface from scaling unchanged. If one passes, return the report anyway; inspect the effect, support and replay before considering unchanged follow-up folds. One-fold success does not establish stability or deployment readiness. Feature engineering remains open.

Save this notebook. Download `nfl_feature_round3_report.zip` from the result directory. Stop the Studio space when finished; do not delete it.